# Sub-etapa 1.4 (complemento) · Control por la duración de las caminatas

En la revisión del 1 de noviembre de 2026 (avance 1.2, punto 4), el profesor guía observó que las caminatas de quienes caen duran en promedio 9,4 s más (48,5 s frente a 39,1 s). El número de pasos ya se había sacado de las variables por eso, pero la duración puede afectar también a otras variables: con un registro más largo hay más pasos y más ciclos, y el CV, la regularidad y el espectro se estiman distinto.

Este notebook hace el control que propuso el profesor: **recortar todas las caminatas a la misma duración y repetir el análisis**. No modifica los notebooks anteriores.

**Antes de ejecutarlo:** el notebook 01 debe estar ejecutado en este computador (genera `datos/procesados/senales_limpias/`, que no se sube a GitHub), y también el 03 (`DATOS = "laboratorio"`) y el 05.

## Plan de análisis (escrito antes de ejecutar)

Este notebook se sube a GitHub **sin ejecutar** antes de correrlo, igual que el 05b.

1. **Duración común.** Cada señal limpia se recorta a sus primeros *D* segundos, donde *D* es la duración útil más corta entre las 72 personas del análisis (se espera que sea de unos 24 s). Así todas las personas quedan con exactamente la misma duración. Si *D* resultara menor que 20 s (el mínimo del notebook 01), se usaría *D* = 20 s y se excluiría a quien no alcance.
2. **Mismas variables.** Las 17 variables se recalculan con **las mismas funciones del notebook 01**, copiadas sin cambios.
3. **Análisis exploratorio.** Mann-Whitney con corrección de Holm para las 17 variables, igual que en la sub-etapa 1.2, y comparación del tamaño del efecto antes y después del recorte.
4. **Modelos, en las mismas 50 particiones del notebook 03:**
   * *Solo frecuencia dominante* (variable fija), comparada con la misma referencia del notebook 03.
   * *1 variable de cámara elegida en cada partición* (el procedimiento del modelo seleccionado), comparada con su resultado en el notebook 05.
   * *Random Forest* con las mismas 14 variables del modelo, comparado con el Random Forest del notebook 03.
   * *Solo duración* (duración original de la caminata): mide cuánto distingue a los grupos la duración por sí sola.
5. **Comparación:** prueba t corregida de Nadeau y Bengio, con Holm sobre las tres comparaciones de antes y después.
6. **Regla de interpretación.** Se concluye que la duración **no explica** el resultado principal si, al recortar, (a) la frecuencia dominante sigue siendo significativa en el análisis exploratorio (p de Holm < 0,05) y (b) el AUC del modelo seleccionado no cambia en más de 0,05 ni de forma significativa. Si no se cumple alguna de las dos, se informa que la duración explica al menos en parte el resultado.

## 1. Configuración

In [ ]:
import platform
import subprocess
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESADOS = RAIZ / "datos" / "procesados"
SENALES = PROCESADOS / "senales_limpias"
RESULTADOS = RAIZ / "resultados"
FIGURAS = RAIZ / "figuras"

SEMILLA = 42
N_PARTICIONES = 5
N_REPETICIONES = 10
FRACCION = 1 / (N_PARTICIONES - 1)
ALFA = 0.05
DURACION_MINIMA_S = 20.0          # mínimo del notebook 01
CAMBIO_MAXIMO_AUC = 0.05          # regla de interpretación (plan, punto 6)
FS = 100                          # frecuencia de muestreo de LTMM
VARIABLES_CAMARA = ["cadencia_pasos_min", "tiempo_paso_s", "cv_tiempo_paso_pct", "tiempo_zancada_s",
                    "cv_tiempo_zancada_pct", "regularidad_paso", "regularidad_zancada", "simetria",
                    "frec_dominante_hz"]
np.random.seed(SEMILLA)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
try:
    commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=RAIZ, capture_output=True,
                            text=True).stdout.strip() or "sin git"
except Exception:
    commit = "sin git"
print(f"Ejecutado: {datetime.now():%Y-%m-%d %H:%M} | commit {commit} | Python {platform.python_version()}")
print("Carpeta de señales limpias:", SENALES, "| existe:", SENALES.is_dir())

## 2. Funciones del notebook 01 (copiadas sin cambios)

In [ ]:
CORTE_PASOS_HZ = 3.0
LIMITES_INTERVALO = (0.5, 1.5)


def pasabajo(x, fs, fc, orden=4):
    b, a = signal.butter(orden, fc / (fs / 2), btype="low")
    return signal.filtfilt(b, a, x, axis=0)


def detectar_pasos(v, fs):
    suave = pasabajo(v, fs, CORTE_PASOS_HZ)
    picos, _ = signal.find_peaks(suave, distance=int(0.3 * fs), prominence=0.3 * np.std(suave))
    return picos, suave


def autocorr(x):
    x = x - x.mean()
    ac = signal.correlate(x, x, mode="full", method="fft")[len(x) - 1:]
    return ac / ac[0]


def regularidad(v, fs):
    ac = autocorr(v)
    lags = np.arange(len(ac)) / fs
    picos, _ = signal.find_peaks(ac)
    candidatos = picos[(lags[picos] > 0.25) & (lags[picos] < 1.0)]
    if len(candidatos) == 0:
        return np.nan, np.nan, np.nan
    p1 = candidatos[np.argmax(ac[candidatos])]
    zona = picos[(picos > 1.5 * p1) & (picos < 2.5 * p1)]
    if len(zona) == 0:
        return ac[p1], np.nan, np.nan
    p2 = zona[np.argmax(ac[zona])]
    return ac[p1], ac[p2], ac[p1] / ac[p2]


def espectro(v, fs):
    f, p = signal.welch(v, fs=fs, nperseg=min(len(v), int(10 * fs)), nfft=8192)
    banda = (f >= 0.5) & (f <= 3.0)
    i = np.argmax(np.where(banda, p, 0))
    mitad = p[i] / 2
    izq = i
    while izq > 0 and p[izq] > mitad:
        izq -= 1
    der = i
    while der < len(p) - 1 and p[der] > mitad:
        der += 1
    return f[i], p[i], f[der] - f[izq]


def intervalos_validos(picos, fs):
    t = np.diff(picos) / fs
    if len(t) < 3:
        return t, np.array([]), np.nan
    ok = (t > LIMITES_INTERVALO[0] * np.median(t)) & (t < LIMITES_INTERVALO[1] * np.median(t))
    zancadas = (t[:-1] + t[1:])[ok[:-1] & ok[1:]]
    return t[ok], zancadas, 100 * (1 - ok.mean())


def variables(nombre, df, fs):
    v = df["acc_v"].to_numpy()
    picos, _ = detectar_pasos(v, fs)
    t_paso, t_zancada, pct_descartado = intervalos_validos(picos, fs)
    d1, d2, sim = regularidad(v, fs)
    f_dom, amp_dom, ancho = espectro(v, fs)
    cv = lambda x: 100 * np.std(x, ddof=1) / np.mean(x) if len(x) > 2 else np.nan
    fila = {
        "registro": nombre,
        "duracion_util_s": len(df) / fs,
        "n_pasos": len(picos),
        "pct_intervalos_descartados": pct_descartado,
        "cadencia_pasos_min": 60 / t_paso.mean() if len(t_paso) else np.nan,
        "tiempo_paso_s": t_paso.mean() if len(t_paso) else np.nan,
        "cv_tiempo_paso_pct": cv(t_paso),
        "tiempo_zancada_s": t_zancada.mean() if len(t_zancada) else np.nan,
        "cv_tiempo_zancada_pct": cv(t_zancada),
        "regularidad_paso": d1,
        "regularidad_zancada": d2,
        "simetria": sim,
        "frec_dominante_hz": f_dom,
        "amplitud_dominante": amp_dom,
        "ancho_pico_hz": ancho,
    }
    for col in df.columns.drop("tiempo_s"):
        fila[f"rms_{col}"] = np.sqrt(np.mean((df[col] - df[col].mean()) ** 2))
    return fila


def holm(p):
    p = np.asarray(p, dtype=float)
    orden = np.argsort(p)
    ajustado, acumulado = np.empty_like(p), 0.0
    for k, i in enumerate(orden):
        acumulado = max(acumulado, min(1.0, (len(p) - k) * p[i]))
        ajustado[i] = acumulado
    return ajustado


def prueba_t_corregida(a, b):
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    J, var = len(d), np.var(d, ddof=1)
    if var == 0:
        return d.mean(), 1.0 if d.mean() == 0 else 0.0
    t = d.mean() / np.sqrt((1 / J + FRACCION) * var)
    return d.mean(), 2 * stats.t.sf(abs(t), df=J - 1)


def intervalo(x):
    x = np.asarray(x, dtype=float)
    J = len(x)
    e = stats.t.ppf(0.975, J - 1) * np.sqrt((1 / J + FRACCION) * np.var(x, ddof=1))
    return x.mean() - e, x.mean() + e

## 3. Recorte a la misma duración y nuevas variables

In [ ]:
lab = pd.read_csv(PROCESADOS / "labwalks_modelo.csv")
completo = pd.read_csv(PROCESADOS / "labwalks_variables.csv").set_index("registro").loc[lab["registro"]]
y = lab["cae"].astype(int).to_numpy()
VARIABLES_MODELO = [c for c in lab.columns if c not in ("registro", "grupo", "cae")]

senales = {}
faltan = []
for r in lab["registro"]:
    ruta = SENALES / f"{r}.csv"
    if ruta.exists():
        senales[r] = pd.read_csv(ruta)
    else:
        faltan.append(r)
if faltan:
    raise FileNotFoundError(f"Faltan {len(faltan)} señales limpias (ej. {faltan[:3]}). Ejecuta primero el notebook 01.")

duraciones = pd.Series({r: len(df) / FS for r, df in senales.items()})
D = float(duraciones.min())
if D < DURACION_MINIMA_S:
    D = DURACION_MINIMA_S
incluidos = duraciones[duraciones >= D].index.tolist()
print(f"Duración común D = {D:.1f} s (la más corta de las {len(duraciones)} caminatas)")
print(f"Personas incluidas: {len(incluidos)} de {len(lab)}")
print("Duración original por grupo (s):")
print(duraciones.groupby(lab.set_index('registro')['grupo']).describe()[["mean", "min", "max"]].round(1))

n = int(round(D * FS))
recortadas = pd.DataFrame(variables(r, senales[r].iloc[:n].reset_index(drop=True), FS) for r in incluidos)
assert np.allclose(recortadas["duracion_util_s"], D, atol=0.01)
NO_SON_VARIABLES = ("registro", "duracion_util_s", "n_pasos", "pct_intervalos_descartados")
VARIABLES = [c for c in recortadas.columns if c not in NO_SON_VARIABLES]
recortadas.to_csv(PROCESADOS / "labwalks_variables_duracion_comun.csv", index=False, float_format="%.5f")
print(f"\n{len(VARIABLES)} variables recalculadas con {D:.1f} s por persona. "
      f"Intervalos descartados: mediana {recortadas['pct_intervalos_descartados'].median():.1f} %, "
      f"máximo {recortadas['pct_intervalos_descartados'].max():.1f} %")

## 4. Análisis exploratorio con la misma duración

In [ ]:
grupo = lab.set_index("registro").loc[incluidos, "cae"].to_numpy()
original = pd.read_csv(PROCESADOS / "labwalks_comparacion_grupos.csv").set_index("variable")
filas = []
for v in VARIABLES:
    a = recortadas.loc[grupo == 0, v].dropna()
    b = recortadas.loc[grupo == 1, v].dropna()
    u, p = stats.mannwhitneyu(b, a, alternative="two-sided")
    rho = stats.spearmanr(recortadas[v], completo.loc[incluidos, v], nan_policy="omit")[0]
    filas.append({"variable": v, "mediana_no_cae": a.median(), "mediana_cae": b.median(),
                  "efecto_r": 2 * u / (len(a) * len(b)) - 1, "p": p,
                  "efecto_r_original": original.loc[v, "efecto_r"] if v in original.index else np.nan,
                  "p_holm_original": original.loc[v, "p_holm"] if v in original.index else np.nan,
                  "rho_con_original": rho})
eda = pd.DataFrame(filas)
eda["p_holm"] = holm(eda["p"])
eda = eda.reindex(eda["efecto_r"].abs().sort_values(ascending=False).index)
eda.to_csv(RESULTADOS / "control_duracion_eda.csv", index=False)
eda[["variable", "mediana_no_cae", "mediana_cae", "efecto_r", "p_holm", "efecto_r_original", "p_holm_original",
     "rho_con_original"]].round(3)

In [ ]:
fig, eje = plt.subplots(figsize=(7.5, 5.5))
orden = eda.sort_values("efecto_r_original")
yy = np.arange(len(orden))
eje.barh(yy - 0.2, orden["efecto_r_original"], height=0.4, color="#9aa5b1", label="duración original")
eje.barh(yy + 0.2, orden["efecto_r"], height=0.4, color="#2a78d6", label=f"todas a {D:.0f} s")
eje.set_yticks(yy, orden["variable"], fontsize=8)
eje.axvline(0, color="#333333", lw=0.8)
eje.set(xlabel="Tamaño del efecto r (negativo = menor en quienes caen)",
        title="Diferencias entre grupos antes y después de igualar la duración")
eje.legend(fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(FIGURAS / "control_duracion_efectos.png", dpi=150)

## 5. Modelos en las mismas 50 particiones

In [ ]:
if len(incluidos) != len(lab):
    print("Atención: no todas las personas alcanzan la duración común; las particiones ya no son las mismas "
          "del notebook 03 y la comparación de a pares no se hace.")
X_rec = recortadas.set_index("registro").loc[lab["registro"]]


def logistica(k=None):
    pasos = [SimpleImputer(strategy="median"), StandardScaler()]
    if k is not None:
        pasos.append(SelectKBest(f_classif, k=k))
    return make_pipeline(*pasos, LogisticRegression(max_iter=1000))


def bosque():
    return make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestClassifier(n_estimators=500, min_samples_leaf=2, class_weight="balanced",
                                                random_state=SEMILLA, n_jobs=-1))


def auc_cv(modelo, X):
    cv = RepeatedStratifiedKFold(n_splits=N_PARTICIONES, n_repeats=N_REPETICIONES, random_state=SEMILLA)
    out = []
    for ent, pru in cv.split(X, y):
        m = clone(modelo).fit(X.iloc[ent], y[ent])
        out.append(roc_auc_score(y[pru], m.predict_proba(X.iloc[pru])[:, 1]))
    return np.array(out)


nuevos = {
    "Solo frecuencia dominante": auc_cv(logistica(), X_rec[["frec_dominante_hz"]]),
    "1 variable de cámara elegida en cada partición": auc_cv(logistica(k=1), X_rec[VARIABLES_CAMARA]),
    "Random Forest": auc_cv(bosque(), X_rec[VARIABLES_MODELO]),
    "Solo duración (original)": auc_cv(logistica(), duraciones.loc[lab["registro"]].to_frame("duracion_s")),
}

previos = pd.read_csv(RESULTADOS / "metricas_por_particion.csv")
previos["particion"] = previos.groupby("modelo").cumcount() + 1
p14 = pd.read_csv(RESULTADOS / "metricas_por_particion_1_4.csv")
ref_eda = next(m for m in previos["modelo"].unique() if m.startswith("Solo ") and m != "Solo TUG")
antes = {
    "Solo frecuencia dominante": previos[previos["modelo"] == ref_eda]["auc"].to_numpy(),
    "1 variable de cámara elegida en cada partición":
        p14[(p14["fuente"] == "laboratorio") & (p14["modelo"] == "1 variable de cámara elegida en cada partición")]["auc"].to_numpy(),
    "Random Forest": previos[previos["modelo"] == "Random Forest"]["auc"].to_numpy(),
}

filas = []
for nombre, a in nuevos.items():
    lo, hi = intervalo(a)
    fila = {"modelo": nombre, "auc_duracion_comun": a.mean(), "ic95": f"[{lo:.2f}; {hi:.2f}]"}
    if nombre in antes and len(antes[nombre]) == len(a):
        dm, p = prueba_t_corregida(a, antes[nombre])
        fila.update({"auc_original": antes[nombre].mean(), "diferencia": dm, "p": p})
    filas.append(fila)
modelos = pd.DataFrame(filas)
m = modelos["p"].notna()
modelos.loc[m, "p_holm"] = holm(modelos.loc[m, "p"])
modelos.to_csv(RESULTADOS / "control_duracion_modelos.csv", index=False)
modelos.round(3)

## 6. Conclusión según la regla del plan

In [ ]:
fd = eda.set_index("variable").loc["frec_dominante_hz"]
sel = modelos.set_index("modelo").loc["1 variable de cámara elegida en cada partición"]
cond_a = fd["p_holm"] < ALFA
cond_b = abs(sel["diferencia"]) <= CAMBIO_MAXIMO_AUC and sel["p_holm"] >= ALFA
fmt = lambda x, k=2: f"{x:.{k}f}".replace(".", ",")
print(f"* Duración común: {fmt(D, 1)} s para las {len(incluidos)} personas.")
print(f"* Frecuencia dominante con la misma duración: r = {fmt(fd['efecto_r'])}, p de Holm = {fmt(fd['p_holm'], 3)} "
      f"(antes: r = {fmt(fd['efecto_r_original'])}, p de Holm = {fmt(fd['p_holm_original'], 3)}).")
sig = eda.loc[eda["p_holm"] < ALFA, "variable"].tolist()
print(f"* Variables significativas tras Holm con la misma duración: {', '.join(sig) if sig else 'ninguna'}.")
print(f"* Modelo seleccionado: AUC {fmt(sel['auc_duracion_comun'])} con la misma duración, frente a "
      f"{fmt(sel['auc_original'])} (diferencia {fmt(sel['diferencia'])}, p de Holm = {fmt(sel['p_holm'], 3)}).")
dur = modelos.set_index("modelo").loc["Solo duración (original)"]
print(f"* La duración original sola distingue a los grupos con AUC {fmt(dur['auc_duracion_comun'])} {dur['ic95'].replace('.', ',')}.")
print("\nConclusión:", "la duración NO explica el resultado principal (se cumplen las dos condiciones del plan)."
      if cond_a and cond_b else
      "la duración explica AL MENOS EN PARTE el resultado (no se cumple alguna condición del plan); se informa así.")

**Interpretación** (complétala con tus palabras después de ejecutar):

* ¿La frecuencia dominante sigue siendo significativa con la misma duración?
* ¿Cambió alguna otra variable, por ejemplo el CV o la regularidad?
* ¿Cuánto distingue la duración por sí sola? Si distingue bien, conviene comentarlo: las personas que caen podrían caminar más lento y por eso tardar más en recorrer el mismo trayecto.